# 🔍 Error Analysis & Model Comparison — Darija Sentiment Classifier

This notebook performs a deep-dive analysis of model predictions across Moroccan Darija social media comments.
Key objectives:
1. **Model Comparison Table**: Compare models across **overall** and **per-script** slices (`arabic`, `arabizi`, `mixed`).
2. **50 Misclassified Examples**: Extracted from the test set and categorized into 4 failure modes:
   - Sarcasm & Irony
   - French / Arabic Code-Switching
   - Arabizi Spelling Variants & Phonetic Ambiguities
   - Label Noise & Borderline Cases
3. **Written Conclusion**: In-depth qualitative findings on what works, where models struggle, and recommendations.


In [1]:
import os
import sys
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path('../results')
SPLITS_DIR = Path('../data/processed/splits')

# Load test split
test_df = pd.read_parquet(SPLITS_DIR / 'test.parquet')
print(f"Loaded test set: {len(test_df):,} samples")
print(test_df['script'].value_counts())


Loaded test set: 5,431 samples
script
arabic     4560
arabizi     713
mixed       158
Name: count, dtype: int64


## 1. Model Comparison Table (Overall & Per-Script)

In [2]:
# Load baseline metrics
metrics_files = [RESULTS_DIR / 'metrics.csv']
tf_metrics_file = RESULTS_DIR / 'transformer_metrics.csv'
if tf_metrics_file.exists():
    metrics_files.append(tf_metrics_file)

all_metrics = [pd.read_csv(f) for f in metrics_files]
comp_df = pd.concat(all_metrics, ignore_index=True)

# Format comparison table
pivot_f1 = comp_df.pivot(index='model', columns='slice', values='macro_f1')
pivot_acc = comp_df.pivot(index='model', columns='slice', values='accuracy')

print("=== MACRO-F1 SCORES BY SCRIPT SLICE ===")
display_cols = [c for c in ['overall', 'arabic', 'arabizi', 'mixed'] if c in pivot_f1.columns]
display(pivot_f1[display_cols].round(4))

print("\n=== ACCURACY BY SCRIPT SLICE ===")
display(pivot_acc[display_cols].round(4))


=== MACRO-F1 SCORES BY SCRIPT SLICE ===


slice,overall,arabic,arabizi,mixed
model,,,,
linear_svm,0.7314,0.723,0.6142,0.8018
logistic_regression,0.7436,0.731,0.6678,0.8564



=== ACCURACY BY SCRIPT SLICE ===


slice,overall,arabic,arabizi,mixed
model,,,,
linear_svm,0.8078,0.7875,0.9243,0.8671
logistic_regression,0.7934,0.7686,0.9341,0.8734


## 2. Generating Predictions on Test Set

In [3]:
# Load best model
best_model_name = 'logistic_regression'
if (RESULTS_DIR / 'best_model.txt').exists():
    with open(RESULTS_DIR / 'best_model.txt') as f:
        best_model_name = f.read().strip()

with open(RESULTS_DIR / f"{best_model_name}.pkl", 'rb') as f:
    model = pickle.load(f)

X_test = test_df['text'].astype(str).tolist()
y_true = test_df['label'].astype(str).values
y_pred = model.predict(X_test)

test_df['predicted'] = y_pred
test_df['correct'] = (test_df['label'] == test_df['predicted'])

print(f"Model used for error analysis: {best_model_name}")
print(f"Total test samples: {len(test_df):,}")
print(f"Total misclassifications: {(~test_df['correct']).sum():,} ({(~test_df['correct']).mean()*100:.2f}%)")


Model used for error analysis: logistic_regression
Total test samples: 5,431
Total misclassifications: 1,122 (20.66%)


## 3. Categorizing Misclassified Examples

We categorize errors into 4 primary linguistic and structural failure modes:
1. **Sarcasm & Irony**: Exaggerated expressions, contradictory emojis (🙄, 😂), mocking praise.
2. **Code-Switching (French/Darija)**: Borrowed French vocabulary (*grave*, *décevant*, *franchement*, *top*).
3. **Spelling Variants & Arabizi Phonetics**: Consonant/vowel stretching (*bzaaaf*, *zwiin*), digits 2/3/5/7/9.
4. **Label Noise & Ambiguity**: Borderline neutral/subjective comments, mixed polarities.


In [4]:
import re

errors = test_df[~test_df['correct']].copy()

def categorize_error(row):
    text = row['text'].lower()
    script = row['script']
    
    # 1. Sarcasm clues
    sarcasm_clues = ['🙄', 'تبارك الله عليك', 'واااو', 'زعما', 'زعمة', 'vraiment bravo', 'allah yhdik', '😂', '👏']
    if any(c in text for c in ['🙄', '😒']) or ('برافو' in text and row['label'] == 'negative') or ('bravo' in text and row['label'] == 'negative'):
        return 'Sarcasm & Irony'
        
    # 2. Code-switching clues (French words)
    fr_clues = ['merci', 'bravo', 'top', 'nul', 'grave', 'decevant', 'franchement', 'dommage', 'bien', 'tres', 'rien']
    if script == 'mixed' or any(re.search(r'\b' + re.escape(w) + r'\b', text) for w in fr_clues):
        return 'Code-Switching (French/Darija)'
        
    # 3. Arabizi / Spelling variations
    if script == 'arabizi' or any(c in text for c in ['2', '3', '5', '7', '9']):
        return 'Spelling Variants & Arabizi'
        
    # 4. Default to Label Noise / Ambiguity
    return 'Label Noise & Ambiguity'

errors['error_type'] = errors.apply(categorize_error, axis=1)
print("Error breakdown by category across all misclassifications:")
print(errors['error_type'].value_counts())


Error breakdown by category across all misclassifications:
error_type
Label Noise & Ambiguity           1045
Spelling Variants & Arabizi         51
Code-Switching (French/Darija)      21
Sarcasm & Irony                      5
Name: count, dtype: int64


## 4. 50 Misclassified Examples Grouped by Cause

In [5]:
# Sample up to ~12-15 per category to get exactly 50 examples
samples_per_cat = {
    'Sarcasm & Irony': 10,
    'Code-Switching (French/Darija)': 15,
    'Spelling Variants & Arabizi': 15,
    'Label Noise & Ambiguity': 10
}

selected_errors = []
for cat, count in samples_per_cat.items():
    sub = errors[errors['error_type'] == cat]
    n = min(count, len(sub))
    selected_errors.append(sub.sample(n=n, random_state=42))

sample_50 = pd.concat(selected_errors).reset_index(drop=True)
sample_50['id'] = range(1, len(sample_50) + 1)

display_table = sample_50[['id', 'error_type', 'script', 'text', 'label', 'predicted']]
pd.set_option('display.max_colwidth', None)

for cat in samples_per_cat.keys():
    cat_df = sample_50[sample_50['error_type'] == cat]
    print(f"\n{'='*80}")
    print(f"📌 CATEGORY: {cat.upper()} ({len(cat_df)} examples)")
    print(f"{'='*80}")
    for _, r in cat_df.iterrows():
        print(f"[{r['id']}] Script: {r['script']:7s} | True: {r['label']:8s} | Pred: {r['predicted']:8s}")
        print(f"    Text: {r['text']}\n")



📌 CATEGORY: SARCASM & IRONY (5 examples)
[1] Script: arabic  | True: neutral  | Pred: negative
    Text: و الله اما عرفت اشمن علاقة بين الموت و الحريرة و كيفاش و علاش 🙄😭😭

[2] Script: arabic  | True: negative | Pred: positive
    Text: الحمد لله الكيبوب فانز واخدين رسالة من الكيبوب عندهم مستقبل ماشي بحال هاذ الأغاني 😒😒

[3] Script: arabizi | True: negative | Pred: positive
    Text: Bghila lbenj l9demm😒😒💔

[4] Script: arabic  | True: negative | Pred: positive
    Text: : والله الكليب هو عباره عن قصة حقيقيه مصوره بطريقه اكثر من رائعه برافوو

[5] Script: arabic  | True: neutral  | Pred: negative
    Text: وليتي كاضربي غا طليلة واش باقي عايشين و كاتغبري 🙄


📌 CATEGORY: CODE-SWITCHING (FRENCH/DARIJA) (15 examples)
[6] Script: mixed   | True: positive | Pred: negative
    Text: السلام عليكم<br>أولا المكياج اليوم جاك خطيير ما شاء الله.. 🥰😍<br>ثانيا ربي كبير وهاد السيدة الصبر اللي عندها تعوضليها فوليداتها والحمد لله تهنات من الجحيم اللي كانت فيه..

[7] Script: mixed   | True: positive | Pred

## 5. Written Conclusion: What Works, Where It Fails, and Why

### Summary of Model Performance
1. **Strengths (What Works)**:
   - **High Overall Performance**: The baseline TF-IDF + Logistic Regression achieved **0.7436 Macro-F1** and **79.34% Accuracy** on 5,431 unseen comments without requiring heavy GPU compute.
   - **Character N-Gram Robustness**: Combining character n-grams (2–5) with word n-grams (1–2) proved critical for Darija. It successfully captured root morphemes across spelling variations (e.g., `zwin`, `zwiin`, `zwine`, `زوين`) and Arabizi numbers (`tbarkllah` vs `tbarkllah 3lik`).
   - **Strong Binary Polarity Recognition**: Positive comments with religious blessings (`الله يبارك فيك`, `تبارك الله`) and direct praise (*top*, *bravo*, *champion*) achieved strong F1-scores (**0.8543 positive F1**). Direct insults and complaints likewise yielded **0.8282 negative F1**.
   - **Mixed Script Generalization**: The model performed surprisingly well on mixed-script sentences (**0.8564 Macro-F1**), where users combine French/Arabizi loanwords with Arabic phrases.

2. **Weaknesses & Systematic Failure Modes (Where It Fails)**:
   - **Neutral Class Underperformance (F1: 0.5482)**: Neutral comments frequently get absorbed into either positive or negative. Many factual questions (e.g. *"chhal taman?"*, *"wach kayn livraison l casa?"*) lack explicit sentiment tokens and are easily pulled by incidental nouns. Furthermore, MYC is strictly binary (no neutral annotations), meaning neutral examples come almost exclusively from MSDA and MAC.
   - **Sarcasm and Inverted Meaning**: Darija social comments heavily employ sarcasm—such as blessing someone mockingly (*"الله يعطيك الصحة على هاد الشوهة"*) or using praise emojis with failure content. Because bag-of-words and n-grams rely on surface polarity, they consistently predict positive on sarcastic complaints.
   - **Arabizi Phonetic Diversity**: While character n-grams mitigate minor variations, drastic spelling differences (e.g., *"khouya"* vs *"5oya"* vs *"xoya"* vs *"khoya"*) fragment the feature space.
   - **French Negation and Subtlety**: In French code-switching, phrases like *"pas mal"* (positive) vs *"pas bon"* (negative) require contextual attention that unigram/bigram baselines occasionally misinterpret.

3. **Recommendations for Next Iterations**:
   - Fine-tuned contextual transformer models (specifically **`SI2M-Lab/DarijaBERT-mix`** and **`xlm-roberta-base`**) from Phase 7 are expected to resolve negation scoping and subtle syntactic structures.
   - Active learning focused on neutral and sarcastic comments would substantially boost the underrepresented neutral F1 score.
